In [ ]:
#!pip install -r /Users/nutthaweecharoenngampis/Developer/sentiment-stock-predictor/requirements.txt

### V1

In [ ]:
"""
nvda_finnhub_impact_tracker.py

- Fetch NVDA company news from Finnhub (Jan 1, 2025 -> Apr 30, 2025)
- Score sentiment with FinBERT (ProsusAI/finbert) on headline+summary
- Fetch daily OHLCV from yfinance (covering lag windows)
- Compute percent changes at multiple day lags and directional accuracy
- Compute correlation/regression per lag and save results + plots
"""

import os
import time
from datetime import datetime, timedelta, date
import pandas as pd
import numpy as np
import finnhub
import yfinance as yf
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from scipy import stats
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from pandas.tseries.offsets import BDay


# -------------------------
# CONFIG
# -------------------------
FINNHUB_API_KEY = 'd2arrjpr01qgk9ugd2ugd2arrjpr01qgk9ugd2v0'
if not FINNHUB_API_KEY:
    raise RuntimeError("Set FINNHUB_API_KEY environment variable before running.")

SYMBOL = "NVDA"
YEAR_START = "2025-01-01"
YEAR_END = "2025-04-30"   # April has 30 days
LAGS_DAYS = [1, 3, 7, 14, 30]   # lags to evaluate (in days)
OUT_DIR = "./nvda_finnhub_impact_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Initialize Finnhub client
# -------------------------
client = finnhub.Client(api_key=FINNHUB_API_KEY)

# -------------------------
# 1) Fetch company news from Finnhub
# -------------------------


print(f"Fetching {SYMBOL} news from {YEAR_START} → {YEAR_END} in weekly batches...")

start = datetime.strptime(YEAR_START, "%Y-%m-%d")
end = datetime.strptime(YEAR_END, "%Y-%m-%d")
delta = timedelta(days=7)

all_news = []

while start <= end:
    batch_end = min(start + delta, end)
    try:
        news_batch = client.company_news(
            SYMBOL,
            _from=start.strftime("%Y-%m-%d"),
            to=batch_end.strftime("%Y-%m-%d")
        )
        if news_batch:
            all_news.extend(news_batch)
            print(f"Fetched {len(news_batch)} items from {start.date()} → {batch_end.date()}")
        else:
            print(f"No items from {start.date()} → {batch_end.date()}")
    except Exception as e:
        print(f"Error fetching news for {start.date()} → {batch_end.date()}: {e}")
    start = batch_end + timedelta(days=1)
    time.sleep(0.25)  # avoid API rate limits

# Convert to DataFrame and clean
df_news = pd.DataFrame(all_news)
if df_news.empty:
    raise RuntimeError("No news returned from Finnhub for the range. Check API key / subscription / ticker.")

df_news = df_news.rename(columns={"datetime": "unix_ts"})

# Drop rows with null or impossible timestamps
df_news = df_news[pd.to_numeric(df_news["unix_ts"], errors="coerce").notna()]
df_news = df_news[df_news["unix_ts"] > 1000000000]  # keep only timestamps after ~2001

# Convert valid timestamps (unit = seconds)
df_news["published_at"] = pd.to_datetime(df_news["unix_ts"], unit="s", utc=True, errors="coerce")
df_news = df_news[df_news["published_at"].notna()]

df_news["pub_date"] = df_news["published_at"].dt.date

df_news["text"] = (
    df_news["headline"].fillna("") + ". " + df_news.get("summary", "").fillna("")
).str.strip()

df_news = df_news[["id", "published_at", "pub_date", "headline", "summary", "url", "text"]]
df_news = df_news.drop_duplicates(subset=["id"]).reset_index(drop=True)

print(f"Total unique valid news items: {len(df_news)}")
print("News date range:", df_news['pub_date'].min(), "→", df_news['pub_date'].max())

news_csv = os.path.join(OUT_DIR, "nvda_news_raw.csv")
df_news.to_csv(news_csv, index=False)
print("Saved raw news ->", news_csv)
# -------------------------
# 2) Sentiment scoring (FinBERT)
# -------------------------
print("Loading FinBERT (ProsusAI/finbert)... this may take a bit.")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert").to(DEVICE)

def sentiment_score_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return (0.0, 0.0, 1.0, 0.0)   # neutral fallback
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        out = model(**inputs)
    probs = softmax(out.logits.cpu().numpy()[0])
    neg, neu, pos = float(probs[0]), float(probs[1]), float(probs[2])
    return (pos - neg, pos, neu, neg)

print("Scoring sentiment for each news item (headline+summary)...")
sent_net = []
sent_pos = []
sent_neu = []
sent_neg = []
for txt in df_news["text"].tolist():
    s_net, p, n, g = sentiment_score_text(txt)
    sent_net.append(s_net)
    sent_pos.append(p)
    sent_neu.append(n)
    sent_neg.append(g)

df_news["sent_net"] = sent_net
df_news["sent_pos"] = sent_pos
df_news["sent_neu"] = sent_neu
df_news["sent_neg"] = sent_neg

sent_csv = os.path.join(OUT_DIR, "nvda_news_with_sentiment.csv")
df_news.to_csv(sent_csv, index=False)
print("Saved news with sentiment ->", sent_csv)

# -------------------------
# 3) Fetch daily prices from yfinance (cover extended window)
# -------------------------
# fetch a little before and after in case lags land outside
start_dt = (pd.to_datetime(YEAR_START) - pd.Timedelta(days=5)).strftime("%Y-%m-%d")
end_dt = (pd.to_datetime(YEAR_END) + pd.Timedelta(days=max(LAGS_DAYS) + 5)).strftime("%Y-%m-%d")

# Latest news date
max_news_date = df_news["pub_date"].max()

# Add 40 business days (more than 30 to be safe)
needed_end_date = max_news_date + BDay(40)


print("Downloading daily prices from yfinance:", start_dt, "→", end_dt)

# Use updated date range here:
df_price = yf.download(SYMBOL, start=start_dt, end=needed_end_date.strftime('%Y-%m-%d'), interval="1d")

# Reset index so Date is a column
df_price.reset_index(inplace=True)

# Flatten multi-index columns if present
if isinstance(df_price.columns, pd.MultiIndex):
    df_price.columns = [col[0].lower() for col in df_price.columns]
else:
    df_price.columns = [col.lower() for col in df_price.columns]

# Rename 'date' column correctly if needed
if "date" not in df_price.columns and "datetime" in df_price.columns:
    df_price.rename(columns={"datetime": "date"}, inplace=True)

# Ensure 'date' column is date type (not datetime)
df_price["date"] = pd.to_datetime(df_price["date"]).dt.date

# Keep only necessary columns and sort
df_price = df_price[["date", "open", "high", "low", "close", "volume"]].sort_values("date").reset_index(drop=True)

prices_csv = os.path.join(OUT_DIR, "nvda_prices_daily.csv")
df_price.to_csv(prices_csv, index=False)
print("Saved daily prices ->", prices_csv)


In [164]:

# -------------------------
# 4) Compute lagged pct changes per news article
# -------------------------
# Ensure price dataframe is sorted and trading_dates list is clean
df_price["date"] = pd.to_datetime(df_price["date"])

df_price = df_price.sort_values("date").reset_index(drop=True)
trading_dates = list(sorted(df_price["date"].unique()))

def get_trading_date_on_or_after(input_date):
    input_date = pd.Timestamp(input_date)
    for d in trading_dates:
        if d >= input_date:
            return d
    return None

def get_close_on_date(date):
    match = df_price.loc[df_price["date"] == date, "close"]
    if len(match) == 0:
        return None
    return float(match.iloc[0])

def add_trading_days(start_date, days):
    if start_date not in trading_dates:
        start_date = get_trading_date_on_or_after(start_date)
    if start_date is None:
        return None
    try:
        start_idx = trading_dates.index(start_date)
    except ValueError:
        return None
    target_idx = start_idx + days
    if target_idx >= len(trading_dates):
        return None
    return trading_dates[target_idx]

records = []
price_min_date = df_price["date"].min()
price_max_date = df_price["date"].max()
print(f"Price data covers from {price_min_date.date()} to {price_max_date.date()}")

for _, r in df_news.iterrows():
    pub_date = pd.to_datetime(r["pub_date"])
    if pub_date < price_min_date:
        print(f"Warning: News date {pub_date.date()} before price data start {price_min_date.date()}. Skipping.")
        continue
    if pub_date > price_max_date:
        print(f"Warning: News date {pub_date.date()} after price data end {price_max_date.date()}. Skipping.")
        continue

    base_td = get_trading_date_on_or_after(pub_date)
    if base_td is None:
        print(f"Warning: No trading date found on/after news date {pub_date.date()}. Skipping.")
        continue
    base_close = get_close_on_date(base_td)
    if base_close is None:
        print(f"Warning: No close price for base trading date {base_td.date()}. Skipping.")
        continue

    rec = {
        "id": r.get("id", None),
        "published_at": r.get("published_at", None),
        "pub_date": pub_date,
        "base_trading_date": base_td,
        "base_close": base_close,
        "sent_net": r["sent_net"],
        "sent_pos": r["sent_pos"],
        "sent_neu": r["sent_neu"],
        "sent_neg": r["sent_neg"],
        "headline": r.get("headline", ""),
        "url": r.get("url", "")
    }

    for lag in LAGS_DAYS:
        target_date = add_trading_days(base_td, lag)
        if target_date is None:
            print(f"Note: Lag {lag}d out of range for article on {pub_date.date()}.")
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        close_price = get_close_on_date(target_date)
        if close_price is None:
            print(f"Warning: No price for lag {lag}d date {target_date.date()} (article {pub_date.date()}).")
            rec[f"close_{lag}d"] = np.nan
            rec[f"pct_change_{lag}d"] = np.nan
            rec[f"trading_date_{lag}d"] = None
            continue

        rec[f"close_{lag}d"] = close_price
        rec[f"pct_change_{lag}d"] = (close_price - base_close) / base_close
        rec[f"trading_date_{lag}d"] = target_date

    # Determine best lag
    lag_changes = {
        lag: rec[f"pct_change_{lag}d"]
        for lag in LAGS_DAYS
        if pd.notna(rec[f"pct_change_{lag}d"])
    }
    if lag_changes:
        best_lag = max(lag_changes, key=lambda d: abs(lag_changes[d]))
        rec["best_lag_days"] = best_lag
        rec["best_lag_pct_change"] = lag_changes[best_lag]
    else:
        rec["best_lag_days"] = np.nan
        rec["best_lag_pct_change"] = np.nan

    records.append(rec)

df_results = pd.DataFrame(records)
df_results.to_csv(os.path.join(OUT_DIR, "nvda_news_lagged_results.csv"), index=False)
print(f"Saved per-article lagged results.")
print(f"Processed articles: {len(df_results)}")
# -------------------------
# 5) Analysis: correlation, regression, directional accuracy
# -------------------------
print("Analyzing results by lag...")
summary_rows = []

for lag in LAGS_DAYS:
    col = f"pct_change_{lag}d"

    # Skip if required columns don't exist
    if "sent_net" not in df_results.columns or col not in df_results.columns:
        print(f"Skipping lag {lag}d — missing 'sent_net' or '{col}' in df_results.")
        continue

    tmp = df_results[["sent_net", col]].dropna()
    n = len(tmp)

    if n >= 5:
        r_val, p_val = stats.pearsonr(tmp["sent_net"], tmp[col])
        X = tmp[["sent_net"]].values.reshape(-1, 1)
        y = tmp[col].values
        lr = LinearRegression().fit(X, y)
        coef = float(lr.coef_[0])
        intercept = float(lr.intercept_)
        r2 = float(lr.score(X, y))
    else:
        r_val = p_val = coef = intercept = r2 = np.nan

    # Directional accuracy
    tmp_dir = tmp.copy()
    tmp_dir = tmp_dir.assign(
        sent_dir=np.sign(tmp_dir["sent_net"]),
        price_dir=np.sign(tmp_dir[col])
    )
    tmp_dir = tmp_dir[tmp_dir["sent_dir"] != 0]
    accuracy = (tmp_dir["sent_dir"] == tmp_dir["price_dir"]).mean() if len(tmp_dir) > 0 else np.nan

    summary_rows.append({
        "lag_days": lag,
        "n": n,
        "pearson_r": r_val,
        "p_value": p_val,
        "reg_coef": coef,
        "reg_intercept": intercept,
        "reg_r2": r2,
        "directional_accuracy": accuracy
    })

df_summary = pd.DataFrame(summary_rows).sort_values("lag_days")

if not df_summary.empty:
    summary_csv = os.path.join(OUT_DIR, "nvda_lag_summary.csv")
    df_summary.to_csv(summary_csv, index=False)
    print("Saved lag summary ->", summary_csv)
    print(df_summary)
else:
    print("No valid lag results to save.")

# -------------------------
# 6) Per-article best lag (largest absolute pct move) and save
# -------------------------
lag_cols = [f"pct_change_{lag}d" for lag in LAGS_DAYS]
print("Lag columns example data:")
print(df_results[[f"pct_change_{lag}d" for lag in LAGS_DAYS]].head())

def best_lag_info(row):
    vals = row[[f"pct_change_{lag}d" for lag in LAGS_DAYS]].abs()
    vals = vals.dropna()
    if vals.empty:
        return (None, np.nan)
    best_col = vals.idxmax()
    best_lag = int(best_col.split("_")[2].replace("d",""))
    best_val = row[best_col]
    return (best_lag, best_val)

df_results["best_lag_days"] = df_results.apply(lambda r: best_lag_info(r)[0], axis=1)
df_results["best_lag_pct_change"] = df_results.apply(lambda r: best_lag_info(r)[1], axis=1)

print("After computing best lag info:")
print(df_results[["best_lag_days", "best_lag_pct_change"]].head(10))
df_results.to_csv(os.path.join(OUT_DIR, "nvda_news_lagged_results_with_bestlag.csv"), index=False)
print("Saved per-article best-lag file.")

# -------------------------
# 7) Plots: correlation vs lag & directional accuracy vs lag
# -------------------------
# correlation plot
plt.figure(figsize=(8,5))
plt.plot(df_summary["lag_days"], df_summary["pearson_r"], marker="o")
plt.xlabel("Lag after news (days)")
plt.ylabel("Pearson correlation (sent_net vs pct_change)")
plt.title("NVDA: sentiment vs pct change correlation by lag")
plt.grid(True)
plt.savefig(os.path.join(OUT_DIR, "correlation_vs_lag.png"), dpi=200)
plt.close()

# directional accuracy bar
plt.figure(figsize=(8,5))
plt.bar(df_summary["lag_days"].astype(str), df_summary["directional_accuracy"])
plt.xlabel("Lag (days)")
plt.ylabel("Directional accuracy (fraction)")
plt.title("Direction accuracy of sentiment prediction by lag")
plt.grid(axis="y")
plt.savefig(os.path.join(OUT_DIR, "directional_accuracy_by_lag.png"), dpi=200)
plt.close()

print("Plots saved in", OUT_DIR)
print("Done.")

Price data covers from 2024-12-27 to 2025-06-24
Saved per-article lagged results.
Processed articles: 3044
Analyzing results by lag...
Saved lag summary -> ./nvda_finnhub_impact_outputs/nvda_lag_summary.csv
   lag_days     n  pearson_r   p_value  reg_coef  reg_intercept    reg_r2  \
0         1  3044  -0.004867  0.788377 -0.000344      -0.004824  0.000024   
1         3  3044   0.042218  0.019840  0.005445      -0.001882  0.001782   
2         7  3044   0.048861  0.007012  0.008447      -0.006879  0.002387   
3        14  3044   0.010055  0.579190  0.002357      -0.003844  0.000101   
4        30  3044   0.026503  0.143774  0.009717       0.044277  0.000702   

   directional_accuracy  
0              0.489488  
1              0.530223  
2              0.535151  
3              0.493101  
4              0.481275  
Lag columns example data:
   pct_change_1d  pct_change_3d  pct_change_7d  pct_change_14d  pct_change_30d
0      -0.029976      -0.059596       0.005139       -0.110342       

In [180]:
import random
import pandas as pd
import os

summary_txt_path = os.path.join(OUT_DIR, "nvda_summary_report.txt")

with open(summary_txt_path, "w", encoding="utf-8") as f:
    def write_and_print(line=""):
        print(line)
        f.write(line + "\n")

    write_and_print("=== NVDA Stock Sentiment Impact Summary ===\n")

    # Check if lag summary exists
    if not df_summary.empty:
        # Find lag with strongest absolute correlation
        best_row = df_summary.loc[df_summary['pearson_r'].abs().idxmax()]
        best_corr_lag = int(best_row["lag_days"])
        best_corr_val = best_row["pearson_r"]

        write_and_print(
            f"Overall, the strongest correlation between news sentiment "
            f"and stock price change is at a lag of {best_corr_lag} trading days."
        )
        write_and_print(f"Correlation coefficient at this lag: {best_corr_val:.3f}\n")
    else:
        write_and_print("No valid lag correlation results to summarize.\n")

    # Filter articles with valid best lag and price impact
    valid_articles = df_results.dropna(subset=["best_lag_days", "best_lag_pct_change"])
    write_and_print(f"Number of articles with valid impact data: {len(valid_articles)}\n")

    if len(valid_articles) == 0:
        write_and_print("No articles with valid impact data to show examples.\n")
    else:
        # Randomly sample up to 5 examples each run
        sample_size = min(5, len(valid_articles))
        examples = valid_articles.sample(sample_size, random_state=None)

        write_and_print(f"Showing {sample_size} example news impacts:\n")

        for _, row in examples.iterrows():
            try:
                best_lag_days = int(row["best_lag_days"])
                trading_date_col = f"trading_date_{best_lag_days}d"
                pct_change_col = f"pct_change_{best_lag_days}d"

                pub_date = pd.to_datetime(row["pub_date"]).date()
                impact_date = row.get(trading_date_col, None)
                impact_pct = row.get(pct_change_col, None)


                write_and_print(f"- Lag days: {pd.to_datetime(impact_date).date() - pub_date}")
                write_and_print(f"  News Date: {pub_date}")

                if pd.notnull(impact_date):
                    write_and_print(f"  Impact Date ({best_lag_days} trading days later): {pd.to_datetime(impact_date).date()}")
                else:
                    write_and_print(f"  Impact Date ({best_lag_days} trading days later): N/A")

                write_and_print(f"  Headline: {row.get('headline', 'N/A')}")

                if pd.notnull(impact_pct):
                    write_and_print(f"  Price Change: {impact_pct*100:.2f}%\n")
                else:
                    write_and_print("  Price Change: N/A (no data)\n")
            except Exception as e:
                write_and_print(f"  Skipped an example due to error: {e}\n")

    write_and_print(
        "Note: Price change is calculated as the percentage change in closing price "
        "relative to the closing price on or just after the news publication date."
    )
    write_and_print(
        "The impact delay (lag) indicates how many trading days after news publication "
        "the stock price shows the strongest per-article reaction."
    )

    write_and_print(f"\nDetailed impact summary saved to: {summary_txt_path}")


=== NVDA Stock Sentiment Impact Summary ===

Overall, the strongest correlation between news sentiment and stock price change is at a lag of 7 trading days.
Correlation coefficient at this lag: 0.049

Number of articles with valid impact data: 3044

Showing 5 example news impacts:

- Lag days: 11 days, 0:00:00
  News Date: 2025-03-27
  Impact Date (7 trading days later): 2025-04-07
  Headline: Bank of America: NVDA Will Ultimately Head Higher, Driven by Very Strong Fundamentals
  Price Change: -12.38%

- Lag days: 1 day, 0:00:00
  News Date: 2025-03-13
  Impact Date (1 trading days later): 2025-03-14
  Headline: Taiwan Semiconductor: Company Is Selling The Shovels To The AI Race
  Price Change: 5.27%

- Lag days: 43 days, 0:00:00
  News Date: 2025-02-06
  Impact Date (30 trading days later): 2025-03-21
  Headline: Most large cryptocurrencies decrease on Solana, XRP drops
  Price Change: -8.52%

- Lag days: 12 days, 0:00:00
  News Date: 2025-02-07
  Impact Date (7 trading days later): 2

In [ ]:
print("News date range:", df_news['pub_date'].min(), "→", df_news['pub_date'].max())

In [ ]:
print(df_results[["pub_date", "pct_change_1d", "pct_change_3d", 
                  "pct_change_7d", "pct_change_14d", "pct_change_30d"]].head(20))

In [186]:
import os
import pandas as pd
import yfinance as yf

OUT_DIR = "./nvda_finnhub_impact_outputs"
summary_txt_path = os.path.join(OUT_DIR, "nvda_summary_report.txt")

# -------------------------
# 1) Parse summary report
# -------------------------
records = []
with open(summary_txt_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    line = line.strip()
    if line.startswith("News Date:"):
        try:
            post_date = pd.to_datetime(line.split(":")[1].strip()).date()
            headline = lines[i+2].strip().replace("Headline: ","")
            price_line = lines[i+3].strip()
            
            if "Price Change" in price_line and "N/A" not in price_line:
                reported_pct = float(price_line.split(":")[1].strip().replace("%","")) / 100
                # Impact date
                impact_line = lines[i+1].strip()
                impact_date_str = impact_line.split(":")[1].strip()
                impact_date = pd.to_datetime(impact_date_str).date()
                records.append({
                    "headline": headline,
                    "posted_date": post_date,
                    "impact_date": impact_date,
                    "reported_pct_change": reported_pct
                })
        except Exception as e:
            print(f"Skipping a line due to parse error: {e}")

df_summary_check = pd.DataFrame(records)
print(f"Parsed {len(df_summary_check)} articles from summary report.")

# -------------------------
# 2) Fetch actual prices from Yahoo Finance
# -------------------------
if not df_summary_check.empty:
    start_dt = min(df_summary_check["posted_date"]) - pd.Timedelta(days=5)
    end_dt = max(df_summary_check["impact_date"]) + pd.Timedelta(days=5)
    
    df_prices = yf.download("NVDA", start=start_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"), interval="1d")
    df_prices.reset_index(inplace=True)
    df_prices["Date"] = pd.to_datetime(df_prices["Date"]).dt.date
    df_prices = df_prices.set_index("Date")
    
    # -------------------------
    # 3) Compute actual percent changes
    # -------------------------
    actual_changes = []
    for _, row in df_summary_check.iterrows():
        post_date = row["posted_date"]
        impact_date = row["impact_date"]
        try:
            close_post = df_prices.loc[post_date, "Close"]
        except KeyError:
            # find next available trading day
            available_dates = sorted([d for d in df_prices.index if d >= post_date])
            if available_dates:
                close_post = df_prices.loc[available_dates[0], "Close"]
            else:
                close_post = None
        
        try:
            close_impact = df_prices.loc[impact_date, "Close"]
        except KeyError:
            # find previous available trading day
            available_dates = sorted([d for d in df_prices.index if d <= impact_date])
            if available_dates:
                close_impact = df_prices.loc[available_dates[-1], "Close"]
            else:
                close_impact = None

        if close_post is not None and close_impact is not None:
            actual_pct = (close_impact - close_post) / close_post
        else:
            actual_pct = None
        
        actual_changes.append(actual_pct)
    
    df_summary_check["actual_pct_change"] = actual_changes
    df_summary_check["diff_pct"] = df_summary_check["actual_pct_change"] - df_summary_check["reported_pct_change"]
    
    print(df_summary_check[["posted_date", "impact_date", "reported_pct_change", "actual_pct_change", "diff_pct"]])
    
    # Save for reference
    df_summary_check.to_csv(os.path.join(OUT_DIR, "nvda_summary_verification.csv"), index=False)
    print("Saved verification results to nvda_summary_verification.csv")
else:
    print("No valid records parsed from summary report.")

/var/folders/76/0r9x2xys7sb0vn337whbttxw0000gn/T/ipykernel_12846/2803312630.py:48: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_prices = yf.download("NVDA", start=start_dt.strftime("%Y-%m-%d"), end=end_dt.strftime("%Y-%m-%d"), interval="1d")
[*********************100%***********************]  1 of 1 completed

Parsed 5 articles from summary report.
  posted_date impact_date  reported_pct_change  \
0  2025-03-27  2025-04-07              -0.1238   
1  2025-03-13  2025-03-14               0.0527   
2  2025-02-06  2025-03-21              -0.0852   
3  2025-02-07  2025-02-19               0.0723   
4  2025-03-03  2025-03-21               0.0320   

                        actual_pct_change  \
0  Ticker
NVDA   -0.123755
dtype: float64   
1  Ticker
NVDA    0.052691
dtype: float64   
2  Ticker
NVDA   -0.085244
dtype: float64   
3   Ticker
NVDA    0.07232
dtype: float64   
4  Ticker
NVDA    0.032008
dtype: float64   

                                 diff_pct  
0  Ticker
NVDA    0.000045
dtype: float64  
1  Ticker
NVDA   -0.000009
dtype: float64  
2  Ticker
NVDA   -0.000044
dtype: float64  
3   Ticker
NVDA    0.00002
dtype: float64  
4  Ticker
NVDA    0.000008
dtype: float64  


Saved verification results to nvda_summary_verification.csv


### Evaluation model

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, mean_absolute_error

OUT_DIR = "./nvda_finnhub_impact_outputs"
results_csv = os.path.join(OUT_DIR, "nvda_news_lagged_results_with_bestlag.csv")
df = pd.read_csv(results_csv)

# --- 1. Helper to assign labels ---
def label_sentiment(x, neutral_thresh=0.05):
    if x > neutral_thresh:
        return 1  # positive
    elif x < -neutral_thresh:
        return -1 # negative
    else:
        return 0  # neutral

def label_price(x, neutral_thresh=0.01):
    if x > neutral_thresh:
        return 1  # up
    elif x < -neutral_thresh:
        return -1 # down
    else:
        return 0  # flat

# --- 2. Apply labels ---
df['sentiment_label'] = df['sent_net'].apply(label_sentiment)
df['impact_label'] = df['best_lag_pct_change'].apply(label_price)

# Keep only rows where we have a real price impact label (non-NaN)
df_eval = df.dropna(subset=['impact_label']).copy()

y_true = df_eval['impact_label']
y_pred = df_eval['sentiment_label']

# --- 3. Classification metrics ---
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=[-1, 0, 1])

# --- 4. Mean Absolute Error on timing (magnitude mismatch) ---
mae = mean_absolute_error(y_true, y_pred)

# --- 5. Backtest: Cumulative return if trading on sentiment ---
capital = 1.0
for _, r in df_eval.iterrows():
    # skip neutral predictions (no trade)
    if r['sentiment_label'] == 0:
        continue
    # take position: +1 long for positive, -1 short for negative
    pos = r['sentiment_label']
    capital *= (1 + pos * r['best_lag_pct_change'])

cumulative_return = capital - 1.0

# --- 6. Save results ---
report_path = os.path.join(OUT_DIR, "nvda_evaluation_metrics.txt")
with open(report_path, "w") as f:
    f.write("=== NVDA Sentiment Model Evaluation ===\n\n")
    f.write(f"Samples evaluated: {len(df_eval)}\n\n")
    f.write(f"Accuracy: {acc:.3f}\n")
    f.write(f"Precision (macro): {prec:.3f}\n")
    f.write(f"Recall (macro): {rec:.3f}\n")
    f.write(f"F1 Score (macro): {f1:.3f}\n")
    f.write(f"Mean Absolute Error (labels): {mae:.3f}\n\n")
    f.write("Confusion Matrix (rows=true, cols=predicted) [order: -1, 0, 1]:\n")
    f.write(str(cm))
    f.write("\n\n")
    f.write(f"Cumulative return from naive trading: {cumulative_return*100:.2f}%\n")

print("=== Evaluation Results ===")
print(f"Accuracy: {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall: {rec:.3f}")
print(f"F1: {f1:.3f}")
print("Confusion matrix:\n", cm)
print(f"MAE: {mae:.3f}")
print(f"Cumulative return: {cumulative_return*100:.2f}%")
print("\nDetailed report saved to", report_path)

=== Evaluation Results ===
Accuracy: 0.386
Precision: 0.342
Recall: 0.263
F1: 0.286
Confusion matrix:
 [[455 323 849]
 [  0   0   0]
 [349 347 721]]
MAE: 1.007
Cumulative return: -99.99%

Detailed report saved to ./nvda_finnhub_impact_outputs/nvda_evaluation_metrics.txt
